# Annotation Overlay Visualization
**Generates per-image overlays** showing bounding boxes, tumour mask boundaries, and class labels.
Each image is saved as an individual PNG inside `results/annotation_overlays/`.

In [1]:
import os, glob
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')          # non-interactive backend — faster, no popups
import matplotlib.pyplot as plt

In [2]:
# === CONFIGURATION ===
PROJECT_DIR = r"c:\Users\miraj\Documents\Research Paper's\Brain Tumor Segmentation and Classification"

IMG_DIR  = os.path.join(PROJECT_DIR, 'data', 'images')
MASK_DIR = os.path.join(PROJECT_DIR, 'data', 'masks')
DET_DIR  = os.path.join(PROJECT_DIR, 'data', 'labels_yolo_det')

OUT_DIR  = os.path.join(PROJECT_DIR, 'results', 'annotation_overlays')
os.makedirs(OUT_DIR, exist_ok=True)

# Class mapping (same as notebook 01)
CLASS_MAP   = {0: 'meningioma', 1: 'glioma', 2: 'pituitary'}
CLASS_COLORS = {
    'meningioma': (255, 100, 100),   # Red
    'glioma':     (0, 170, 255),     # Orange-blue
    'pituitary':  (0, 255, 0),       # Green
}
BOUNDARY_COLOR = (255, 255, 0)       # Yellow for tumour boundary

print(f'Images  : {IMG_DIR}')
print(f'Masks   : {MASK_DIR}')
print(f'Labels  : {DET_DIR}')
print(f'Output  : {OUT_DIR}')

Images  : c:\Users\miraj\Documents\Research Paper's\Brain Tumor Segmentation and Classification\data\images
Masks   : c:\Users\miraj\Documents\Research Paper's\Brain Tumor Segmentation and Classification\data\masks
Labels  : c:\Users\miraj\Documents\Research Paper's\Brain Tumor Segmentation and Classification\data\labels_yolo_det
Output  : c:\Users\miraj\Documents\Research Paper's\Brain Tumor Segmentation and Classification\results\annotation_overlays


In [3]:
# Gather all image filenames (stem only, no extension)
image_paths = sorted(glob.glob(os.path.join(IMG_DIR, '*.png')))
fnames = [os.path.splitext(os.path.basename(p))[0] for p in image_paths]
print(f'Found {len(fnames)} images to process.')

Found 3064 images to process.


### Show annotations for all images in the segmentation dataset

In [2]:
"""
Create side-by-side comparison images:

    LEFT  = Original MRI image
    RIGHT = Annotated MRI image with YOLO-segmentation polygon boundaries

Class names are displayed on the annotated image:
    Meningioma
    Glioma
    Pituitary

Dataset structure:
    dataset_v3/
        train/images/*.jpg
        train/labels/*.txt
        val/images/*.jpg
        val/labels/*.txt
        test/images/*.jpg
        test/labels/*.txt
        data.yaml
"""

import os
import cv2
import numpy as np
from glob import glob
from pathlib import Path

# ============================================================
# CONFIGURATION
# ============================================================

PROJECT_DIR = r"c:\Users\miraj\Documents\Research Paper's\Brain Tumor Segmentation and Classification"

DATASET_DIR = os.path.join(PROJECT_DIR, "dataset_v3")

# Output folder
OUT_DIR = os.path.join(
    PROJECT_DIR,
    "results",
    "annotation_comparison"
)

os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# CLASS MAPPING
# ============================================================

CLASS_MAP = {
    0: "Meningioma",
    1: "Glioma",
    2: "Pituitary"
}

# OpenCV uses BGR format
CLASS_COLORS = {
    "Meningioma": (255, 100, 100),
    "Glioma":     (0, 170, 255),
    "Pituitary":  (0, 255, 0)
}

BOUNDARY_COLOR = (255, 255, 0)

# ============================================================
# SETTINGS
# ============================================================

SPLITS = ["train", "val", "test"]

IMG_EXTS = (
    ".png",
    ".jpg",
    ".jpeg",
    ".tif",
    ".tiff",
    ".bmp"
)

LINE_THICKNESS = 2

# Text settings
FONT = cv2.FONT_HERSHEY_SIMPLEX
CLASS_FONT_SCALE = 0.65
TITLE_FONT_SCALE = 0.9
TEXT_THICKNESS = 2

# ============================================================
# PRINT CONFIGURATION
# ============================================================

print("=" * 70)
print("YOLO SEGMENTATION ANNOTATION COMPARISON")
print("=" * 70)

print(f"Dataset : {DATASET_DIR}")
print(f"Output  : {OUT_DIR}")

# ============================================================
# CHECK DATA.YAML
# ============================================================

yaml_path = os.path.join(DATASET_DIR, "data.yaml")

if os.path.exists(yaml_path):

    with open(yaml_path, "r") as f:
        yaml_content = f.read()

    print("\n--- data.yaml ---")
    print(yaml_content)
    print("--- End data.yaml ---\n")

# ============================================================
# PARSE YOLO SEGMENTATION LABEL
# ============================================================

def parse_yolo_seg_label(label_path, img_w, img_h):
    """
    Parse YOLO segmentation labels.

    Format:
        class_id x1 y1 x2 y2 x3 y3 ...

    Coordinates are normalized between 0 and 1.

    Returns:
        [(class_id, polygon_points), ...]
    """

    polygons = []

    with open(label_path, "r") as f:

        for line_no, line in enumerate(f, 1):

            parts = line.strip().split()

            if not parts:
                continue

            try:
                class_id = int(float(parts[0]))
                coords = [float(v) for v in parts[1:]]

            except ValueError:
                print(
                    f"[warn] {label_path} line {line_no}: "
                    f"invalid values -- skipped"
                )
                continue

            # Minimum 3 points = 6 coordinates
            if len(coords) < 6 or len(coords) % 2 != 0:

                print(
                    f"[warn] {label_path} line {line_no}: "
                    f"invalid polygon -- skipped"
                )

                continue

            # Convert coordinates to numpy
            pts = np.array(
                coords,
                dtype=np.float32
            ).reshape(-1, 2)

            # Normalize coordinates -> pixel coordinates
            pts[:, 0] *= img_w
            pts[:, 1] *= img_h

            pts = pts.astype(np.int32)

            polygons.append(
                (class_id, pts)
            )

    return polygons


# ============================================================
# DRAW POLYGON BOUNDARIES
# ============================================================

def draw_boundaries(image_bgr, polygons):
    """
    Draw polygon boundaries and class names.

    No polygon filling.
    """

    annotated = image_bgr.copy()

    for object_index, (class_id, pts) in enumerate(polygons):

        class_name = CLASS_MAP.get(
            class_id,
            f"Unknown ({class_id})"
        )

        color = CLASS_COLORS.get(
            class_name,
            BOUNDARY_COLOR
        )

        # ----------------------------------------------------
        # Draw polygon boundary
        # ----------------------------------------------------

        cv2.polylines(
            annotated,
            [pts],
            isClosed=True,
            color=color,
            thickness=LINE_THICKNESS
        )

        # ----------------------------------------------------
        # Find a suitable position for class name
        # ----------------------------------------------------

        x = int(np.min(pts[:, 0]))
        y = int(np.min(pts[:, 1])) - 10

        # Keep text inside image
        if y < 25:
            y = int(np.min(pts[:, 1])) + 25

        # ----------------------------------------------------
        # Draw black background behind text
        # ----------------------------------------------------

        (text_w, text_h), baseline = cv2.getTextSize(
            class_name,
            FONT,
            CLASS_FONT_SCALE,
            TEXT_THICKNESS
        )

        cv2.rectangle(
            annotated,
            (x, y - text_h - baseline),
            (x + text_w + 8, y + 5),
            (0, 0, 0),
            -1
        )

        # ----------------------------------------------------
        # Draw class name
        # ----------------------------------------------------

        cv2.putText(
            annotated,
            class_name,
            (x + 4, y),
            FONT,
            CLASS_FONT_SCALE,
            color,
            TEXT_THICKNESS,
            cv2.LINE_AA
        )

    return annotated


# ============================================================
# ADD LEGEND
# ============================================================

def add_legend(image):
    """
    Add class legend to the top-left of the annotated image.
    """

    legend_items = [
        ("Meningioma", CLASS_COLORS["Meningioma"]),
        ("Glioma", CLASS_COLORS["Glioma"]),
        ("Pituitary", CLASS_COLORS["Pituitary"])
    ]

    x_start = 15
    y_start = 35

    for i, (class_name, color) in enumerate(legend_items):

        y = y_start + (i * 35)

        # Color box
        cv2.rectangle(
            image,
            (x_start, y - 20),
            (x_start + 22, y + 2),
            color,
            -1
        )

        # Class name
        cv2.putText(
            image,
            class_name,
            (x_start + 32, y),
            FONT,
            0.6,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

    return image


# ============================================================
# ADD TITLE
# ============================================================

def add_title(image, title):
    """
    Add title at the top of an image.
    """

    result = image.copy()

    # Create space at top
    title_height = 45

    result = cv2.copyMakeBorder(
        result,
        title_height,
        0,
        0,
        0,
        cv2.BORDER_CONSTANT,
        value=(0, 0, 0)
    )

    cv2.putText(
        result,
        title,
        (15, 30),
        FONT,
        TITLE_FONT_SCALE,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )

    return result


# ============================================================
# CREATE SIDE-BY-SIDE IMAGE
# ============================================================

def create_comparison(original, annotated):
    """
    Create:

        ORIGINAL | ANNOTATED

    Both images have the same size.
    """

    # Make both images same height
    h1, w1 = original.shape[:2]
    h2, w2 = annotated.shape[:2]

    target_height = max(h1, h2)

    def resize_to_height(img, target_h):

        h, w = img.shape[:2]

        scale = target_h / h

        new_w = int(w * scale)

        return cv2.resize(
            img,
            (new_w, target_h),
            interpolation=cv2.INTER_AREA
        )

    original = resize_to_height(
        original,
        target_height
    )

    annotated = resize_to_height(
        annotated,
        target_height
    )

    # Add titles
    original = add_title(
        original,
        "ORIGINAL IMAGE"
    )

    annotated = add_title(
        annotated,
        "ANNOTATED IMAGE"
    )

    # Make heights equal after title
    max_height = max(
        original.shape[0],
        annotated.shape[0]
    )

    def pad_height(img, height):

        difference = height - img.shape[0]

        if difference <= 0:
            return img

        return cv2.copyMakeBorder(
            img,
            0,
            difference,
            0,
            0,
            cv2.BORDER_CONSTANT,
            value=(0, 0, 0)
        )

    original = pad_height(
        original,
        max_height
    )

    annotated = pad_height(
        annotated,
        max_height
    )

    # Combine horizontally
    comparison = np.hstack(
        (original, annotated)
    )

    return comparison


# ============================================================
# PROCESS ONE SPLIT
# ============================================================

def process_split(split):

    img_split_dir = os.path.join(
        DATASET_DIR,
        split,
        "images"
    )

    label_split_dir = os.path.join(
        DATASET_DIR,
        split,
        "labels"
    )

    out_split_dir = os.path.join(
        OUT_DIR,
        split
    )

    if not os.path.isdir(img_split_dir):

        print(
            f'[skip] No images folder for "{split}"'
        )

        return 0

    os.makedirs(
        out_split_dir,
        exist_ok=True
    )

    img_paths = [
        p
        for p in sorted(
            glob(
                os.path.join(
                    img_split_dir,
                    "*"
                )
            )
        )
        if p.lower().endswith(IMG_EXTS)
    ]

    saved = 0

    # ========================================================
    # PROCESS IMAGES
    # ========================================================

    for img_path in img_paths:

        stem = Path(img_path).stem

        label_path = os.path.join(
            label_split_dir,
            stem + ".txt"
        )

        # ----------------------------------------------------
        # Read image
        # ----------------------------------------------------

        image = cv2.imread(
            img_path,
            cv2.IMREAD_COLOR
        )

        if image is None:

            print(
                f"[warn] Could not read image: {img_path}"
            )

            continue

        h, w = image.shape[:2]

        # ----------------------------------------------------
        # Read labels
        # ----------------------------------------------------

        if os.path.exists(label_path):

            polygons = parse_yolo_seg_label(
                label_path,
                w,
                h
            )

        else:

            print(
                f"[info] No label file for {stem} "
                f"(split={split})"
            )

            polygons = []

        # ----------------------------------------------------
        # Create annotated image
        # ----------------------------------------------------

        annotated = draw_boundaries(
            image,
            polygons
        )

        # ----------------------------------------------------
        # Add class legend
        # ----------------------------------------------------

        annotated = add_legend(
            annotated
        )

        # ----------------------------------------------------
        # Create ORIGINAL + ANNOTATED
        # ----------------------------------------------------

        comparison = create_comparison(
            image,
            annotated
        )

        # ----------------------------------------------------
        # Save combined image
        # ----------------------------------------------------

        out_path = os.path.join(
            out_split_dir,
            f"{stem}_comparison.png"
        )

        success = cv2.imwrite(
            out_path,
            comparison
        )

        if success:

            saved += 1

        else:

            print(
                f"[warn] Could not save: {out_path}"
            )

    print(
        f"[{split}] saved "
        f"{saved}/{len(img_paths)} comparison images "
        f"-> {out_split_dir}"
    )

    return saved


# ============================================================
# MAIN
# ============================================================

def main():

    total = 0

    for split in SPLITS:

        total += process_split(
            split
        )

    print("\n" + "=" * 70)
    print(
        f"Done. Total comparison images saved: {total}"
    )
    print("=" * 70)

    print(
        f"\nOutput folder:\n{OUT_DIR}"
    )


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    main()


YOLO SEGMENTATION ANNOTATION COMPARISON
Dataset : c:\Users\miraj\Documents\Research Paper's\Brain Tumor Segmentation and Classification\dataset_v3
Output  : c:\Users\miraj\Documents\Research Paper's\Brain Tumor Segmentation and Classification\results\annotation_comparison

--- data.yaml ---
names:
- glioma
- meningioma
- pituitary
nc: 3
path: /content/drive/Othercomputers/My Laptop/Documents/Research Paper's/Brain Tumor
  Segmentation and Classification/dataset_v3
test: test/images
train: train/images
val: val/images

--- End data.yaml ---

[train] saved 2912/2912 comparison images -> c:\Users\miraj\Documents\Research Paper's\Brain Tumor Segmentation and Classification\results\annotation_comparison\train
[val] saved 92/92 comparison images -> c:\Users\miraj\Documents\Research Paper's\Brain Tumor Segmentation and Classification\results\annotation_comparison\val
[test] saved 60/60 comparison images -> c:\Users\miraj\Documents\Research Paper's\Brain Tumor Segmentation and Classification\r

In [3]:
"""
YOLO SEGMENTATION ANNOTATION COMPARISON

Creates one combined image for every MRI:

                    image_name.jpg
        ┌──────────────────┬──────────────────┐
        │ ORIGINAL IMAGE   │ ANNOTATED IMAGE  │
        │                  │                  │
        │      MRI         │   Polygon        │
        │                  │   boundary       │
        │                  │   Glioma         │
        └──────────────────┴──────────────────┘

Features:
    - Original image on LEFT
    - Annotated image on RIGHT
    - Polygon boundary only (NO FILL)
    - Class names displayed
    - Image filename displayed ONLY ONCE
    - Processes train / val / test
    - Saves comparison images as PNG

Dataset structure:

    dataset_v3/
        train/
            images/
            labels/
        val/
            images/
            labels/
        test/
            images/
            labels/
        data.yaml
"""

import os
import cv2
import numpy as np
from glob import glob
from pathlib import Path


# ============================================================
# CONFIGURATION
# ============================================================

PROJECT_DIR = r"c:\Users\miraj\Documents\Research Paper's\Brain Tumor Segmentation and Classification"

DATASET_DIR = os.path.join(
    PROJECT_DIR,
    "dataset_v3"
)

# Output directory
OUT_DIR = os.path.join(
    PROJECT_DIR,
    "results",
    "annotation_comparison"
)

os.makedirs(
    OUT_DIR,
    exist_ok=True
)


# ============================================================
# CLASS MAPPING
# ============================================================

# Must match data.yaml class order

CLASS_MAP = {
    0: "Meningioma",
    1: "Glioma",
    2: "Pituitary"
}


# ============================================================
# CLASS COLORS
# ============================================================

# OpenCV uses BGR format

CLASS_COLORS = {
    "Meningioma": (255, 100, 100),
    "Glioma":     (0, 170, 255),
    "Pituitary":  (0, 255, 0)
}

# Fallback color
BOUNDARY_COLOR = (255, 255, 0)


# ============================================================
# SETTINGS
# ============================================================

SPLITS = [
    "train",
    "val",
    "test"
]

IMG_EXTS = (
    ".png",
    ".jpg",
    ".jpeg",
    ".tif",
    ".tiff",
    ".bmp"
)

# Polygon boundary thickness
LINE_THICKNESS = 2

# Class-name font
FONT = cv2.FONT_HERSHEY_SIMPLEX

CLASS_FONT_SCALE = 0.65
CLASS_TEXT_THICKNESS = 2

# Filename font
FILENAME_FONT_SCALE = 0.75
FILENAME_TEXT_THICKNESS = 2

# Top space for filename
TITLE_HEIGHT = 55


# ============================================================
# PRINT CONFIGURATION
# ============================================================

print("=" * 75)
print("YOLO SEGMENTATION ANNOTATION COMPARISON")
print("=" * 75)

print(f"Dataset : {DATASET_DIR}")
print(f"Output  : {OUT_DIR}")


# ============================================================
# CHECK DATA.YAML
# ============================================================

yaml_path = os.path.join(
    DATASET_DIR,
    "data.yaml"
)

if os.path.exists(yaml_path):

    print("\n--- data.yaml ---")

    with open(
        yaml_path,
        "r",
        encoding="utf-8"
    ) as f:

        print(f.read())

    print("--- End data.yaml ---\n")

else:

    print(
        f"[warning] data.yaml not found: {yaml_path}"
    )


# ============================================================
# PARSE YOLO SEGMENTATION LABEL
# ============================================================

def parse_yolo_seg_label(
    label_path,
    img_w,
    img_h
):
    """
    Parse a YOLO segmentation label file.

    YOLO segmentation format:

        class_id x1 y1 x2 y2 x3 y3 ... xn yn

    Coordinates are normalized between 0 and 1.

    Returns:

        [
            (class_id, polygon_points),
            ...
        ]
    """

    polygons = []

    with open(
        label_path,
        "r",
        encoding="utf-8"
    ) as f:

        for line_no, line in enumerate(f, 1):

            parts = line.strip().split()

            # Skip empty lines
            if not parts:
                continue

            # ------------------------------------------------
            # Read class ID
            # ------------------------------------------------

            try:

                class_id = int(
                    float(parts[0])
                )

                coords = [
                    float(value)
                    for value in parts[1:]
                ]

            except ValueError:

                print(
                    f"[warning] {label_path} "
                    f"line {line_no}: invalid values"
                )

                continue

            # ------------------------------------------------
            # Check polygon
            # ------------------------------------------------

            # Minimum 3 points = 6 coordinates

            if (
                len(coords) < 6
                or len(coords) % 2 != 0
            ):

                print(
                    f"[warning] {label_path} "
                    f"line {line_no}: invalid polygon"
                )

                continue

            # ------------------------------------------------
            # Convert coordinates to numpy
            # ------------------------------------------------

            pts = np.array(
                coords,
                dtype=np.float32
            ).reshape(-1, 2)

            # ------------------------------------------------
            # Convert normalized coordinates
            # to pixel coordinates
            # ------------------------------------------------

            pts[:, 0] *= img_w
            pts[:, 1] *= img_h

            # Keep points inside image
            pts[:, 0] = np.clip(
                pts[:, 0],
                0,
                img_w - 1
            )

            pts[:, 1] = np.clip(
                pts[:, 1],
                0,
                img_h - 1
            )

            pts = pts.astype(
                np.int32
            )

            polygons.append(
                (
                    class_id,
                    pts
                )
            )

    return polygons


# ============================================================
# DRAW POLYGON BOUNDARIES + CLASS NAMES
# ============================================================

def draw_boundaries(
    image_bgr,
    polygons
):
    """
    Draw polygon boundaries and class names.

    IMPORTANT:
        Polygon is NOT filled.
    """

    annotated = image_bgr.copy()

    for class_id, pts in polygons:

        # ----------------------------------------------------
        # Get class name
        # ----------------------------------------------------

        class_name = CLASS_MAP.get(
            class_id,
            f"Unknown ({class_id})"
        )

        # ----------------------------------------------------
        # Get class color
        # ----------------------------------------------------

        color = CLASS_COLORS.get(
            class_name,
            BOUNDARY_COLOR
        )

        # ----------------------------------------------------
        # Draw ONLY polygon boundary
        # ----------------------------------------------------

        cv2.polylines(
            annotated,
            [pts],
            isClosed=True,
            color=color,
            thickness=LINE_THICKNESS,
            lineType=cv2.LINE_AA
        )

        # ----------------------------------------------------
        # Find position for class name
        # ----------------------------------------------------

        x = int(
            np.min(pts[:, 0])
        )

        y = int(
            np.min(pts[:, 1])
        ) - 10

        # Keep text inside image
        if y < 30:

            y = int(
                np.min(pts[:, 1])
            ) + 30

        # ----------------------------------------------------
        # Get text size
        # ----------------------------------------------------

        (
            text_w,
            text_h
        ), baseline = cv2.getTextSize(
            class_name,
            FONT,
            CLASS_FONT_SCALE,
            CLASS_TEXT_THICKNESS
        )

        # ----------------------------------------------------
        # Draw black background
        # behind class name
        # ----------------------------------------------------

        cv2.rectangle(
            annotated,
            (
                x,
                y - text_h - baseline - 4
            ),
            (
                x + text_w + 8,
                y + 5
            ),
            (0, 0, 0),
            -1
        )

        # ----------------------------------------------------
        # Draw class name
        # ----------------------------------------------------

        cv2.putText(
            annotated,
            class_name,
            (
                x + 4,
                y
            ),
            FONT,
            CLASS_FONT_SCALE,
            color,
            CLASS_TEXT_THICKNESS,
            cv2.LINE_AA
        )

    return annotated


# ============================================================
# ADD CLASS LEGEND
# ============================================================

def add_legend(image):
    """
    Add a class legend to the annotated image.
    """

    legend_items = [
        (
            "Meningioma",
            CLASS_COLORS["Meningioma"]
        ),
        (
            "Glioma",
            CLASS_COLORS["Glioma"]
        ),
        (
            "Pituitary",
            CLASS_COLORS["Pituitary"]
        )
    ]

    x_start = 15
    y_start = 35

    for index, (
        class_name,
        color
    ) in enumerate(legend_items):

        y = (
            y_start
            + index * 32
        )

        # Color box
        cv2.rectangle(
            image,
            (
                x_start,
                y - 18
            ),
            (
                x_start + 20,
                y + 2
            ),
            color,
            -1
        )

        # Class name
        cv2.putText(
            image,
            class_name,
            (
                x_start + 28,
                y
            ),
            FONT,
            0.55,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

    return image


# ============================================================
# RESIZE IMAGE TO SAME HEIGHT
# ============================================================

def resize_to_height(
    image,
    target_height
):

    h, w = image.shape[:2]

    scale = (
        target_height / h
    )

    new_width = int(
        w * scale
    )

    return cv2.resize(
        image,
        (
            new_width,
            target_height
        ),
        interpolation=cv2.INTER_AREA
    )


# ============================================================
# CREATE COMPARISON IMAGE
# ============================================================

def create_comparison(
    original,
    annotated,
    image_name
):
    """
    Create one combined image:

                     image_name.jpg

        ┌────────────────────┬────────────────────┐
        │  ORIGINAL IMAGE    │  ANNOTATED IMAGE   │
        │                    │                    │
        │       MRI          │     Polygon        │
        │                    │     boundary       │
        │                    │     Class name     │
        └────────────────────┴────────────────────┘

    Filename is displayed ONLY ONCE.
    """

    # --------------------------------------------------------
    # Make both images same height
    # --------------------------------------------------------

    h1 = original.shape[0]
    h2 = annotated.shape[0]

    target_height = max(
        h1,
        h2
    )

    original = resize_to_height(
        original,
        target_height
    )

    annotated = resize_to_height(
        annotated,
        target_height
    )

    # --------------------------------------------------------
    # Add panel labels
    # --------------------------------------------------------

    def add_panel_label(
        image,
        label
    ):

        result = image.copy()

        label_height = 40

        result = cv2.copyMakeBorder(
            result,
            label_height,
            0,
            0,
            0,
            cv2.BORDER_CONSTANT,
            value=(0, 0, 0)
        )

        cv2.putText(
            result,
            label,
            (15, 28),
            FONT,
            0.7,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

        return result

    original = add_panel_label(
        original,
        "ORIGINAL IMAGE"
    )

    annotated = add_panel_label(
        annotated,
        "ANNOTATED IMAGE"
    )

    # --------------------------------------------------------
    # Make same height
    # --------------------------------------------------------

    max_height = max(
        original.shape[0],
        annotated.shape[0]
    )

    def pad_height(
        image,
        height
    ):

        difference = (
            height - image.shape[0]
        )

        if difference <= 0:
            return image

        return cv2.copyMakeBorder(
            image,
            0,
            difference,
            0,
            0,
            cv2.BORDER_CONSTANT,
            value=(0, 0, 0)
        )

    original = pad_height(
        original,
        max_height
    )

    annotated = pad_height(
        annotated,
        max_height
    )

    # --------------------------------------------------------
    # Combine horizontally
    # --------------------------------------------------------

    comparison = np.hstack(
        (
            original,
            annotated
        )
    )

    # --------------------------------------------------------
    # Add space for filename
    # --------------------------------------------------------

    comparison = cv2.copyMakeBorder(
        comparison,
        TITLE_HEIGHT,
        0,
        0,
        0,
        cv2.BORDER_CONSTANT,
        value=(0, 0, 0)
    )

    # --------------------------------------------------------
    # Calculate filename position
    # --------------------------------------------------------

    (
        text_w,
        text_h
    ), baseline = cv2.getTextSize(
        image_name,
        FONT,
        FILENAME_FONT_SCALE,
        FILENAME_TEXT_THICKNESS
    )

    image_width = comparison.shape[1]

    text_x = (
        image_width - text_w
    ) // 2

    text_y = 38

    # --------------------------------------------------------
    # Draw filename ONLY ONCE
    # --------------------------------------------------------

    cv2.putText(
        comparison,
        image_name,
        (
            text_x,
            text_y
        ),
        FONT,
        FILENAME_FONT_SCALE,
        (255, 255, 255),
        FILENAME_TEXT_THICKNESS,
        cv2.LINE_AA
    )

    return comparison


# ============================================================
# PROCESS ONE DATASET SPLIT
# ============================================================

def process_split(split):

    # --------------------------------------------------------
    # Input directories
    # --------------------------------------------------------

    img_split_dir = os.path.join(
        DATASET_DIR,
        split,
        "images"
    )

    label_split_dir = os.path.join(
        DATASET_DIR,
        split,
        "labels"
    )

    # --------------------------------------------------------
    # Output directory
    # --------------------------------------------------------

    out_split_dir = os.path.join(
        OUT_DIR,
        split
    )

    # --------------------------------------------------------
    # Check image directory
    # --------------------------------------------------------

    if not os.path.isdir(
        img_split_dir
    ):

        print(
            f'[skip] Images folder not found '
            f'for "{split}": {img_split_dir}'
        )

        return 0

    os.makedirs(
        out_split_dir,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Find images
    # --------------------------------------------------------

    img_paths = [
        p
        for p in sorted(
            glob(
                os.path.join(
                    img_split_dir,
                    "*"
                )
            )
        )
        if p.lower().endswith(
            IMG_EXTS
        )
    ]

    print("\n" + "-" * 75)
    print(
        f"Processing split: {split}"
    )
    print(
        f"Images found: {len(img_paths)}"
    )
    print("-" * 75)

    saved = 0

    # ========================================================
    # PROCESS EACH IMAGE
    # ========================================================

    for index, img_path in enumerate(
        img_paths,
        1
    ):

        # ----------------------------------------------------
        # Get image filename
        # ----------------------------------------------------

        stem = Path(
            img_path
        ).stem

        image_name = os.path.basename(
            img_path
        )

        # ----------------------------------------------------
        # Label path
        # ----------------------------------------------------

        label_path = os.path.join(
            label_split_dir,
            stem + ".txt"
        )

        # ----------------------------------------------------
        # Read image
        # ----------------------------------------------------

        image = cv2.imread(
            img_path,
            cv2.IMREAD_COLOR
        )

        if image is None:

            print(
                f"[warning] Could not read: "
                f"{img_path}"
            )

            continue

        h, w = image.shape[:2]

        # ----------------------------------------------------
        # Read segmentation labels
        # ----------------------------------------------------

        if os.path.exists(
            label_path
        ):

            polygons = parse_yolo_seg_label(
                label_path,
                w,
                h
            )

        else:

            print(
                f"[info] No label file for: "
                f"{image_name}"
            )

            polygons = []

        # ----------------------------------------------------
        # Create annotated image
        # ----------------------------------------------------

        annotated = draw_boundaries(
            image,
            polygons
        )

        # ----------------------------------------------------
        # Add legend
        # ----------------------------------------------------

        annotated = add_legend(
            annotated
        )

        # ----------------------------------------------------
        # Create side-by-side comparison
        # ----------------------------------------------------

        comparison = create_comparison(
            image,
            annotated,
            image_name
        )

        # ----------------------------------------------------
        # Output path
        # ----------------------------------------------------

        out_path = os.path.join(
            out_split_dir,
            f"{stem}_comparison.png"
        )

        # ----------------------------------------------------
        # Save
        # ----------------------------------------------------

        success = cv2.imwrite(
            out_path,
            comparison
        )

        if success:

            saved += 1

        else:

            print(
                f"[warning] Failed to save: "
                f"{out_path}"
            )

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (
            index % 100 == 0
            or index == len(img_paths)
        ):

            print(
                f"Progress: "
                f"{index}/{len(img_paths)}"
            )

    # --------------------------------------------------------
    # Split summary
    # --------------------------------------------------------

    print(
        f"\n[{split}] "
        f"Saved {saved}/{len(img_paths)} "
        f"comparison images"
    )

    print(
        f"Output: {out_split_dir}"
    )

    return saved


# ============================================================
# MAIN FUNCTION
# ============================================================

def main():

    total = 0

    # --------------------------------------------------------
    # Process train / val / test
    # --------------------------------------------------------

    for split in SPLITS:

        total += process_split(
            split
        )

    # --------------------------------------------------------
    # Final summary
    # --------------------------------------------------------

    print("\n")
    print("=" * 75)
    print("PROCESSING COMPLETED")
    print("=" * 75)

    print(
        f"Total comparison images saved: {total}"
    )

    print(
        f"\nOutput directory:\n{OUT_DIR}"
    )

    print("\nStructure:")

    for split in SPLITS:

        print(
            f"  {split}/"
        )

    print("=" * 75)


# ============================================================
# RUN PROGRAM
# ============================================================

if __name__ == "__main__":

    main()

YOLO SEGMENTATION ANNOTATION COMPARISON
Dataset : c:\Users\miraj\Documents\Research Paper's\Brain Tumor Segmentation and Classification\dataset_v3
Output  : c:\Users\miraj\Documents\Research Paper's\Brain Tumor Segmentation and Classification\results\annotation_comparison

--- data.yaml ---
names:
- glioma
- meningioma
- pituitary
nc: 3
path: /content/drive/Othercomputers/My Laptop/Documents/Research Paper's/Brain Tumor
  Segmentation and Classification/dataset_v3
test: test/images
train: train/images
val: val/images

--- End data.yaml ---


---------------------------------------------------------------------------
Processing split: train
Images found: 2912
---------------------------------------------------------------------------
Progress: 100/2912
Progress: 200/2912
Progress: 300/2912
Progress: 400/2912
Progress: 500/2912
Progress: 600/2912
Progress: 700/2912
Progress: 800/2912
Progress: 900/2912
Progress: 1000/2912
Progress: 1100/2912
Progress: 1200/2912
Progress: 1300/2912
Progre